<a href="https://colab.research.google.com/github/bsenst/llm-zoomcamp/blob/bsenst/llm-zoomcamp-code/week04/04-evaluation-homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 02 Homework

## Homework Setup

First, we need to install the required libraries. Since `uv` is mentioned in the homework description, we will use `pip` to install it and then use `uv` to manage the rest of the dependencies as per the homework instructions.

In [ ]:
# Install uv for dependency management (if not already installed in Colab's base environment)
# This is retained as per original homework setup, though subsequent installs will use pip.
!pip install uv

In [ ]:
# Install the required dependencies using pip.
# Note: Some of these might already be present in Colab's base environment.
!pip install onnxruntime tokenizers numpy tqdm minsearch gitsource huggingface-hub

Next, we need to download two helper scripts: `download.py` (fetches an ONNX model) and `embedder.py` (the `Embedder` class).

In [ ]:
import os

PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

# Download download.py
!wget {PREFIX}/download.py -O download.py

# Download embedder.py
!wget {PREFIX}/embedder.py -O embedder.py

Finally, we run `download.py` to fetch the default ONNX model (`Xenova/all-MiniLM-L6-v2`).

In [ ]:
!python download.py

---

## Q1. Embedding a query

Let's embed the query: "How does approximate nearest neighbor search work?" and find the first value (`v[0]`) of the resulting 384-number vector.

In [ ]:
from embedder import Embedder

# Initialize the embedder
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

# Get the first value of the vector
first_value = query_vector[0]
print(f"The first value of the query vector is: {first_value:.2f}")

---

## Q2. Cosine similarity

Now, let's load the data from the course repository, embed the content of `02-vector-search/lessons/07-sqlitesearch-vector.md`, and compute its cosine similarity with the query vector from Q1.

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Find the specific document for Q2
sqlitesearch_doc = None
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        sqlitesearch_doc = doc
        break

if sqlitesearch_doc is None:
    raise ValueError("Document '02-vector-search/lessons/07-sqlitesearch-vector.md' not found.")

# Embed the content of the found document
doc_vector = embedder.encode(sqlitesearch_doc['content'])

# Calculate cosine similarity (dot product for normalized vectors)
cosine_similarity = query_vector.dot(doc_vector)

print(f"Cosine similarity with '02-vector-search/lessons/07-sqlitesearch-vector.md': {cosine_similarity:.2f}")

---

## Q3. Chunking and search by hand

Now we'll chunk the documents, embed each chunk, and score the Q1 query against all chunks to find the highest-scoring chunk's filename.

In [ ]:
from gitsource import chunk_documents
import numpy as np

# Chunk the documents
chunks = chunk_documents(documents, size=2000, step=1000)

# Embed every chunk's content
chunk_contents = [chunk['content'] for chunk in chunks]

# The embedder.encode_batch method can be used to process multiple texts at once
# It returns a numpy array of embeddings
X = embedder.encode_batch(chunk_contents)

# Ensure query_vector is a 1D array for dot product
v = query_vector.flatten()

# Score the Q1 query against all chunks
scores = X.dot(v)

# Find the index of the highest-scoring chunk
highest_score_idx = np.argmax(scores)

# Get the highest-scoring chunk
highest_scoring_chunk = chunks[highest_score_idx]

print(f"The highest-scoring chunk belongs to the file: {highest_scoring_chunk['filename']}")

---

## Q4. Vector search with minsearch

We will use `VectorSearch` from `minsearch` to run a search for the query: "What metric do we use to evaluate a search engine?" and find the filename of the first result.

In [ ]:
from minsearch import VectorSearch

# Create VectorSearch index
vector_search_engine = VectorSearch()

# Index the chunks using the pre-computed embeddings X and the original chunks as payload.
# According to the minsearch source code, fit expects two arguments: 'vectors' and 'payload'.
# The 'id' field is automatically added by minsearch if not present, and embeddings are taken from 'vectors'.
vector_search_engine.fit(vectors=X, payload=chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
# Embed the query for vector search
query_q4_vector = embedder.encode(query_q4)

# Perform the search
# The search method in minsearch does not take an embedding_model argument directly; it expects the query vector.
results_q4 = vector_search_engine.search(query_q4_vector, num_results=1)

if results_q4:
    first_result_filename = results_q4[0]['filename']
    print(f"The filename of the first result for Q4 is: {first_result_filename}")
else:
    print("No results found for Q4.")

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. We'll index the same chunks with `Index` from `minsearch`, using `content` as a text field.

Then, we'll run both searches for the query: "How do I store vectors in PostgreSQL?" and take the top 5 results from each method. Finally, we'll identify which file shows up in the vector results but not in the text results.

In [ ]:
from minsearch import Index

# Initialize minsearch.Index for keyword search
# Use 'content' as a text field, as specified
keyword_search_engine = Index(text_fields=['content'], keyword_fields=['filename'])

# Fit the index with the chunks
keyword_search_engine.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"

# Perform keyword search (top 5 results)
keyword_results_q5 = keyword_search_engine.search(query=query_q5, num_results=5)

print("Keyword Search Results (Top 5 Filenames):")
keyword_filenames = []
for r in keyword_results_q5:
    print(f"- {r['filename']}")
    keyword_filenames.append(r['filename'])

# Perform vector search (top 5 results) using the embedder and vector_search_engine from Q4
query_q5_vector = embedder.encode(query_q5)
vector_results_q5 = vector_search_engine.search(query_q5_vector, num_results=5)

print("\nVector Search Results (Top 5 Filenames):")
vector_filenames = []
for r in vector_results_q5:
    print(f"- {r['filename']}")
    vector_filenames.append(r['filename'])

# Identify which file shows up in vector results but not in text results
vector_only_files = []
for filename in vector_filenames:
    if filename not in keyword_filenames:
        vector_only_files.append(filename)

if vector_only_files:
    print("\nFile(s) in Vector Results but not in Keyword Results:")
    for filename in vector_only_files:
        print(f"- {filename}")
else:
    print("\nNo files found in vector results that are not in keyword results.")

## Q6. Hybrid search with Reciprocal Rank Fusion (RRF)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query_q6 = "How do I give the model access to tools?"

# Perform keyword search (top results for RRF)
# For RRF, it's generally good to get more results than the final num_results
keyword_results_q6 = keyword_search_engine.search(query=query_q6, num_results=10)

# Perform vector search (top results for RRF)
query_q6_vector = embedder.encode(query_q6)
vector_results_q6 = vector_search_engine.search(query_q6_vector, num_results=10)

# Fuse the results using RRF
fused_results = rrf([vector_results_q6, keyword_results_q6], num_results=1)

if fused_results:
    first_ranked_file = fused_results[0]['filename']
    print(f"The file ranked first after RRF is: {first_ranked_file}")
else:
    print("No results after RRF.")

## Summary & Lessons learned

* Importance of Embeddings: Embed text queries and document content into numerical vectors, which is fundamental for semantic search.
* Vector Similarity: Cosine similarity can be used to measure the semantic relatedness between a query vector and document vectors.
* Document Chunking: Chunking documents for effective vector search, especially with larger texts, to pinpoint relevant sections.
* Strengths and Weaknesses of Search Types: Vector search excels at finding semantically similar content even with different phrasing, while keyword search is strong for exact term matching. Each has its own use cases.
* Hybrid Search with RRF: The power of hybrid search, specifically using Reciprocal Rank Fusion (RRF), to combine the best aspects of both vector and keyword search. RRF helps to surface documents that are relevant by both semantic meaning and keyword presence, often leading to more robust search results.

# Week 04 Homework

In [ ]:
! pip install openai pydantic python-dotenv pandas

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [ ]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

## Q1. Generating questions

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# Ensure the OpenRouter API key is set as an environment variable (e.g., in Colab secrets)
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

target_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

total_input_tokens = 0
num_calls = 0

print("Generating questions and collecting token usage...")

# Filter the 'documents' list for the target filenames
selected_documents = [doc for doc in documents if doc['filename'] in target_filenames]

for doc in selected_documents:
    print(f"Processing {doc['filename']}...")
    prompt = f"""
Based on the following document content, generate a concise question that can be answered directly from the text.

Document Content:
---
{doc['content']}
---

Question:
"""
    try:
        response = client.chat.completions.create(
            # Use a model available on OpenRouter that is equivalent to gpt-4o-mini
            # Example: 'openai/gpt-4o-mini' or similar if OpenRouter supports it directly
            # Otherwise, you might need to choose a different model supported by OpenRouter
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )
        input_tokens = response.usage.prompt_tokens
        total_input_tokens += input_tokens
        num_calls += 1
        print(f"  Input tokens for {doc['filename']}: {input_tokens}")
    except Exception as e:
        print(f"  Error processing {doc['filename']}: {e}")
        print("  Please ensure your OPENROUTER_API_KEY is set correctly as an environment variable.")

if num_calls > 0:
    average_input_tokens = total_input_tokens / num_calls
    print(f"\nTotal input tokens across {num_calls} calls: {total_input_tokens}")
    print(f"Average input tokens per call: {average_input_tokens:.2f}")

    # Comparing to the given options
    options = [140, 1400, 14000, 140000]
    closest_option = min(options, key=lambda x: abs(x - average_input_tokens))
    print(f"The closest option is: {closest_option}")
else:
    print("\nNo documents processed. Ensure target filenames are correct and API key is set.")


In [ ]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

## Q2. First result with text search

In [ ]:
import pandas as pd
from gitsource import chunk_documents
from minsearch import Index, VectorSearch
import numpy as np

# Load ground-truth.csv with pandas into a list of records
# The file was already downloaded by a previous cell.
ground_truth = pd.read_csv('ground-truth.csv').to_dict(orient='records')
print(f"Loaded {len(ground_truth)} ground truth questions.")

# Create chunks from the documents
# 'documents' variable is available from previous cells (e.g., from Q2 data loading)
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks.")

# Initialize minsearch.Index for keyword search
keyword_search_engine = Index(text_fields=['content'], keyword_fields=['filename'])
keyword_search_engine.fit(chunks)

# Initialize minsearch.VectorSearch for vector search
# 'embedder' object and 'X' (embeddings of chunks) are available from previous cells
vector_search_engine = VectorSearch()
vector_search_engine.fit(vectors=X, payload=chunks)

# Define text_search function
def text_search(query, num_results=5):
    return keyword_search_engine.search(query=query, num_results=num_results)

# Define vector_search function
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_search_engine.search(query_vector, num_results=num_results)

# Define rrf function (provided by user)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# Define hybrid_search function
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

# Q2. First result with text search
q = ground_truth[0]["question"]
print(f"\nFirst question from ground truth: {q}")

# Run text_search for the first question
text_search_results_q2 = text_search(q, num_results=1)

if text_search_results_q2:
    first_result_filename_q2 = text_search_results_q2[0]['filename']
    print(f"The filename of the first result for Q2 with text search is: {first_result_filename_q2}")
else:
    print("No results found for Q2 with text search.")


## Q3. First result with vector search

In [ ]:
# Q3. First result with vector search
# The question 'q' is already defined from ground_truth[0]["question"]

# Run vector_search for the first question
vector_search_results_q3 = vector_search(q, num_results=1)

if vector_search_results_q3:
    first_result_filename_q3 = vector_search_results_q3[0]['filename']
    print(f"The filename of the first result for Q3 with vector search is: {first_result_filename_q3}")
else:
    print("No results found for Q3 with vector search.")


## Q4. Evaluating text search

In [ ]:
from tqdm.auto import tqdm

def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break
    return total_score / len(relevance)

def evaluate(search_function, ground_truth):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q['filename'] # Use 'filename' from ground_truth
        results = search_function(q['question'])
        # Ensure 'd' has 'filename' attribute, as it comes from text_search which returns chunks
        relevance = [int(d['filename'] == doc_id) for d in results]
        relevance_total.append(relevance)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

# Define a search function that takes a query and returns results
# This wrapper is needed because evaluate expects a function that directly takes a query
def search_text(query):
    return text_search(query, num_results=5)

# Evaluate text_search on the ground truth data
eval_results_text_search = evaluate(search_text, ground_truth)

hit_rate_text_search = eval_results_text_search['hit_rate']
print(f"Hit Rate for text_search: {hit_rate_text_search:.2f}")


## Q5. Evaluating vector search

In [ ]:
# Define a search function for vector search that takes a query and returns results
def search_vector(query):
    return vector_search(query, num_results=5)

# Evaluate vector_search on the ground truth data
eval_results_vector_search = evaluate(search_vector, ground_truth)

mrr_vector_search = eval_results_vector_search['mrr']
print(f"MRR for vector_search: {mrr_vector_search:.2f}")


## Q6. Tuning hybrid search

In [ ]:
k_values = [1, 50, 100, 200]

mrr_results = {}

for k in k_values:
    print(f"Evaluating hybrid search with k={k}...")
    # Define a search function that takes a query and returns results
    # This wrapper is needed because evaluate expects a function that directly takes a query
    def search_hybrid(query):
        # hybrid_search already returns top 5 results by default if num_results not passed
        return hybrid_search(query, k=k)

    # Evaluate hybrid_search on the ground truth data
    eval_results_hybrid_search = evaluate(search_hybrid, ground_truth)

    mrr_hybrid_search = eval_results_hybrid_search['mrr']
    mrr_results[k] = mrr_hybrid_search
    print(f"  MRR for hybrid_search with k={k}: {mrr_hybrid_search:.2f}")

best_k = None
best_mrr = -1

# Find the k with the best MRR, picking the smallest k in case of a tie
for k, mrr_value in mrr_results.items():
    if mrr_value > best_mrr:
        best_mrr = mrr_value
        best_k = k
    elif mrr_value == best_mrr and k < best_k:
        best_k = k

print(f"\nThe k value that gives the best MRR is: {best_k} with an MRR of {best_mrr:.2f}")
